# 🌊 Tutorial: Differentiable Ocean Dynamics with Reactant.jl

## 1. Introduction
Intuitive explanation...

In [ ]:
using Pkg; Pkg.activate("."); Pkg.instantiate()
using Reactant, Enzyme, CUDA, MPI, Oceananigans
using Oceananigans.Architectures: ReactantState
include("ocean_utils.jl")

MPI.Init()
CUDA.versioninfo()

## 2. Setting the Scene


In [ ]:
const Nx = 48; const Ny = 96; const Nz = 32
const Lx = 1e6; const Ly = 2e6
k_center = collect(1:Nz); Δz_center = @. 10 * 1.104^(Nz - k_center)
const Lz = sum(Δz_center); z_faces = vcat([-Lz], -Lz .+ cumsum(Δz_center)); z_faces[Nz+1] = 0
parameters = (Ly=Ly, Lz=Lz, ΔT=8, h=1000.0, y_sponge=1.9e6, λt=7days, μ=1/30days, Lx=Lx, Nz=Nz)

## 3. Creating the Ocean Architecture


In [ ]:
architecture = ReactantState()
grid = make_grid(architecture, Nx, Ny, Nz, Lx, Ly, Lz, z_faces, 4)
model = build_model(grid, 2.5minutes, parameters)
@info "Built "

## 4. Visualizing results


In [ ]:
using CairoMakie
fig = Figure(size = (800, 400))
ax = Axis(fig[1, 1], title = "Surface Temperature")
heatmap!(ax, Array(interior(model.tracers.T, :, :, Nz)), colormap = :thermal)
fig